# Conventional Approach — XGBoost with Optuna

Loads the train and test sets written by `src/split_data.py`, tunes an XGBoost model with Optuna (a Bayesian TPE search) using **time-ordered cross-validation inside the training set**, fits the best parameters on the whole training set, and evaluates **once** on the held-out test set.

All logic is in `src/xgboost_tuning.py`; this notebook calls it and shows the results. Run `python src/split_data.py` first if `train.csv` and `test.csv` are missing.

In [ ]:
import sys; print(sys.executable)

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd

SRC = Path.cwd().parent / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from xgboost_tuning import (N_TRIALS, RANDOM_SEED, RESULTS_FILE, THRESHOLD, TRAIN_FRACTIONS,
                            check_folds, evaluate, fit_final, load_split, save_results,
                            time_ordered_folds, tune)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

## 1. Load train and test

`first_date` and `churn` are separated from the features. `first_date` is kept only to order the training rows for the CV folds; it is never fed to the model.

In [ ]:
X_train, y_train, dates_train, X_test, y_test = load_split()

assert list(X_train.columns) == list(X_test.columns), 'train/test features differ'
assert 'first_date' not in X_train.columns and 'churn' not in X_train.columns

print(f'Train : {X_train.shape}  churn rate {y_train.mean():.3f}  '
      f'({dates_train.min().date()} -> {dates_train.max().date()})')
print(f'Test  : {X_test.shape}  churn rate {y_test.mean():.3f}')
nan_cols = X_train.isna().sum()
print(f'\nFeatures with missing values in train: {(nan_cols > 0).sum()} '
      f'(XGBoost handles NaN natively)')
print(f'Non-numeric feature columns: {list(X_train.select_dtypes(exclude="number").columns)}')

In [ ]:
[i for i in (X_train.columns)]

## 2. Time-ordered cross-validation folds

The training customers are ordered by `first_date`. Fold `0.6` trains on the earliest 60% and validates on the remaining, latest 40%; `0.7` trains on the earliest 70% and validates on the latest 30%; likewise `0.8` and `0.9`. Validation windows overlap and all end at the last training date, but every validation customer is later than every training customer in the same fold.

In [ ]:
folds = time_ordered_folds(dates_train)
check_folds(dates_train, folds)

dates = dates_train.reset_index(drop=True)
rows = []
for fraction, (train_idx, val_idx) in zip(TRAIN_FRACTIONS, folds):
    rows.append({
        'fold (train share)': fraction,
        'train rows': len(train_idx),
        'validation rows': len(val_idx),
        'train: last date': dates.iloc[train_idx].max().date(),
        'validation: first date': dates.iloc[val_idx].min().date(),
        'validation churn rate': y_train.reset_index(drop=True).iloc[val_idx].mean(),
    })
pd.DataFrame(rows).round(3)

## 3. Hyperparameter tuning with Optuna

Optuna's TPE sampler (seeded with `RANDOM_SEED = 42`) runs `N_TRIALS = 100` trials. Each trial fits one model per fold and its score is the **mean validation AUC-ROC over the 4 folds**. The search space is in `suggest_params` in `src/xgboost_tuning.py`.

In [ ]:
study = tune(X_train, y_train, folds, n_trials=N_TRIALS, seed=RANDOM_SEED, show_progress_bar=True)

print(f'\nBest mean CV AUC-ROC : {study.best_value:.4f}  (trial {study.best_trial.number})')
print('Per-fold AUC-ROC     :', dict(zip(TRAIN_FRACTIONS,
      [round(a, 4) for a in study.best_trial.user_attrs["fold_aucs"]])))
print('Best parameters      :')
for name, value in study.best_params.items():
    print(f'  {name:<18}: {value:.6g}' if isinstance(value, float) else f'  {name:<18}: {value}')

## 4. Fit the final model on the whole training set

In [ ]:
final_model = fit_final(study.best_params, X_train, y_train)
print(f'Final model fitted on {len(X_train):,} training customers, {X_train.shape[1]} features')

## 5. Evaluate on the held-out test set

Threshold-based metrics use a probability threshold of 0.5; AUC-ROC is threshold-free. The test set is used only here.

In [ ]:
metrics = evaluate(final_model, X_test, y_test)

summary = pd.Series({k: metrics[k] for k in ('accuracy', 'precision', 'recall', 'f1', 'auc_roc')},
                    name='test set').round(4)
print(f'=== Test set results (threshold {THRESHOLD}, churn = positive class) ===')
print(summary.to_string())

print('\nConfusion matrix')
display(pd.DataFrame([[metrics['tn'], metrics['fp']], [metrics['fn'], metrics['tp']]],
                     index=pd.Index(['actual retained', 'actual churned'], name=''),
                     columns=['predicted retained', 'predicted churned']))

print(f'Mean CV AUC-ROC {study.best_value:.4f} vs test AUC-ROC {metrics["auc_roc"]:.4f}')

## 6. Save results

Metrics, best parameters and CV details go to `results/conventional_xgboost.json`.

In [ ]:
out_path = save_results(study, metrics, X_train, X_test, folds)
print(f'Results saved to {out_path.relative_to(out_path.parents[1])}')